In [ ]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,243 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd6

In [ ]:
!sudo apt-get update
!sudo apt-get install -y mpich mpich-doc libmpich-dev gcc

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
%%writefile tp1.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    printf("Hello from process %d out of %d\n", rank, size);

    MPI_Finalize();
    return 0;
}


Writing tp1.c


In [ ]:
!mpicc -o tp1 tp1.c


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./tp1


Hello from process 1 out of 4
Hello from process 0 out of 4
Hello from process 3 out of 4
Hello from process 2 out of 4


In [ ]:
%%writefile tp2.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int data;

    if (rank == 0) {
        data = 10;
        MPI_Ssend(&data, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
        printf("Process %d sent data: %d\n", rank, data);
    } else if (rank == 1) {
        printf("Process %d received data: %d\n", rank, data);
    }

    MPI_Finalize();
    return 0;
}


Overwriting tp2.c


In [ ]:
!mpicc -o tp2 tp2.c


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./tp2


Process 1 received data: 0


In [ ]:
%%writefile tp3.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    int data;

    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (rank == 0) {
        data = 10;
        printf("Process %d broadcasting data: %d\n", rank, data);
    }

    MPI_Bcast(&data, 1, MPI_INT, 0, MPI_COMM_WORLD);

    printf("Process %d received data: %d\n", rank, data);

    MPI_Finalize();
    return 0;
}


Writing tp3.c


In [ ]:
!mpicc -o tp3 tp3.c


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./tp3


Process 0 broadcasting data: 99
Process 0 received data: 99
Process 1 received data: 99
Process 3 received data: 99
Process 2 received data: 99


In [ ]:
%%writefile sum_array_mpi.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>

int main(int argc, char *argv[]) {
    int rank, size, N, i;
    int *T = NULL;          // Main array on the master process
    int *local_T = NULL;    // Sub-array for each process
    int local_size;         // Size of each sub-array
    int local_sum = 0;      // Local sum for each process
    int *partial_sums = NULL; // Array to store partial sums on master
    int total_sum = 0;      // Final total sum

    // Initialize MPI
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // Get process rank
    MPI_Comm_size(MPI_COMM_WORLD, &size); // Get number of processes

    // Master process sets the array size and initializes the array
    if (rank == 0) {
        N = 8; // Example: Set N to 8 (must be divisible by size)

        // Check if N is divisible by the number of processes
        if (N % size != 0) {
            printf("Error: N must be divisible by the number of processes (%d)\n", size);
            MPI_Abort(MPI_COMM_WORLD, 1);
        }

        // Allocate and initialize the array T
        T = (int *)malloc(N * sizeof(int));
        printf("Initializing array T with values...\n");
        for (i = 0; i < N; i++) {
            T[i] = i + 1; // Fill array with 1, 2, 3, ..., N
            printf("%d ", T[i]);
        }
        printf("\n");
    }

    // Broadcast the size N to all processes
    MPI_Bcast(&N, 1, MPI_INT, 0, MPI_COMM_WORLD);

    // Compute the size of the sub-array for each process
    local_size = N / size;

    // Allocate memory for the local sub-array on each process
    local_T = (int *)malloc(local_size * sizeof(int));

    // Scatter the array T to all processes
    MPI_Scatter(T, local_size, MPI_INT, local_T, local_size, MPI_INT, 0, MPI_COMM_WORLD);

    // Each process computes the local sum of its sub-array
    local_sum = 0;
    for (i = 0; i < local_size; i++) {
        local_sum += local_T[i];
    }
    printf("Process %d: Local sum = %d\n", rank, local_sum);

    // Master process allocates memory to store partial sums
    if (rank == 0) {
        partial_sums = (int *)malloc(size * sizeof(int));
    }

    // Gather the partial sums from all processes to the master
    MPI_Gather(&local_sum, 1, MPI_INT, partial_sums, 1, MPI_INT, 0, MPI_COMM_WORLD);

    // Master process computes the final total sum
    if (rank == 0) {
        total_sum = 0;
        for (i = 0; i < size; i++) {
            total_sum += partial_sums[i];
        }
        printf("Total sum of the array = %d\n", total_sum);

        // Free allocated memory
        free(T);
        free(partial_sums);
    }

    // Free local array on all processes
    free(local_T);

    // Finalize MPI
    MPI_Finalize();
    return 0;
}

Writing sum_array_mpi.c


In [ ]:
!mpicc -o sum_array_mpi sum_array_mpi.c

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./sum_array_mpi

Initializing array T with values...
1 2 3 4 5 6 7 8 
Process 0: Local sum = 3
Process 2: Local sum = 11
Process 1: Local sum = 7
Process 3: Local sum = 15
Total sum of the array = 36


# EXO 6


In [ ]:
%%writefile dot_product_mpi.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>

int main(int argc, char *argv[]) {
    int rank, size, N, i;
    int *A = NULL, *B = NULL;
    int *local_A = NULL, *local_B = NULL;
    int local_size;
    int local_dot_product = 0;
    int final_dot_product = 0;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (rank == 0) {
        N = 8;
        if (N % size != 0) {
            printf("Error: N must be divisible by the number of processes (%d)\n", size);
            MPI_Abort(MPI_COMM_WORLD, 1);
        }
        A = (int *)malloc(N * sizeof(int));
        B = (int *)malloc(N * sizeof(int));
        printf("Initializing array A with values...\n");
        for (i = 0; i < N; i++) {
            A[i] = i + 1;
            printf("%d ", A[i]);
        }
        printf("\n");
        printf("Initializing array B with values...\n");
        for (i = 0; i < N; i++) {
            B[i] = (i + 1) * 2;
            printf("%d ", B[i]);
        }
        printf("\n");
    }

    MPI_Bcast(&N, 1, MPI_INT, 0, MPI_COMM_WORLD);
    local_size = N / size;
    local_A = (int *)malloc(local_size * sizeof(int));
    local_B = (int *)malloc(local_size * sizeof(int));
    MPI_Scatter(A, local_size, MPI_INT, local_A, local_size, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Scatter(B, local_size, MPI_INT, local_B, local_size, MPI_INT, 0, MPI_COMM_WORLD);

    local_dot_product = 0;
    for (i = 0; i < local_size; i++) {
        local_dot_product += local_A[i] * local_B[i];
    }
    printf("Process %d: Local dot product = %d\n", rank, local_dot_product);

    MPI_Reduce(&local_dot_product, &final_dot_product, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("Final dot product = %d\n", final_dot_product);
        free(A);
        free(B);
    }

    free(local_A);
    free(local_B);
    MPI_Finalize();
    return 0;
}

Writing dot_product_mpi.c


In [ ]:
!mpicc -o dot_product_mpi dot_product_mpi.c

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./dot_product_mpi

Initializing array A with values...
1 2 3 4 5 6 7 8 
Initializing array B with values...
2 4 6 8 10 12 14 16 
Process 0: Local dot product = 10
Process 2: Local dot product = 122
Process 3: Local dot product = 226
Process 1: Local dot product = 50
Final dot product = 408
